# Guardrail

Các guardrail giúp bạn xây dựng các ứng dụng AI an toàn và tuân thủ quy định thông qua việc xác thực và lọc nội dung tại những điểm trọng yếu trong quá trình thực thi của agent. Chúng có thể phát hiện thông tin nhạy cảm, thực thi các chính sách nội dung, kiểm chứng đầu ra và ngăn chặn các hành vi thiếu an toàn trước khi chúng gây ra sự cố.

Các trường hợp sử dụng phổ biến bao gồm:

* Ngăn chặn rò rỉ thông tin nhận dạng cá nhân (PII)
* Phát hiện và ngăn chặn các cuộc tấn công prompt injection
* Chặn nội dung độc hại hoặc không phù hợp
* Đảm bảo tuân thủ các quy tắc nghiệp vụ và yêu cầu pháp lý
* Kiểm chứng độ chính xác và chất lượng của đầu ra

Bạn có thể triển khai guardrail bằng cách sử dụng [middleware](https://docs.langchain.com/oss/python/langchain/middleware) để can thiệp vào luồng thực thi tại các điểm chiến lược - trước khi agent bắt đầu, sau khi hoàn thành, hoặc xung quanh các lệnh gọi model và tool.

<p align="center">
    <img src="https://mintcdn.com/langchain-5e9cc07a/RAP6mjwE5G00xYsA/oss/images/middleware_final.png?fit=max&auto=format&n=RAP6mjwE5G00xYsA&q=85&s=eb4404b137edec6f6f0c8ccb8323eaf1" height="500">
</p>

Các guardrail có thể được triển khai theo hai phương pháp hỗ trợ lẫn nhau:

* **Guardrail tất định**: Sử dụng logic dựa trên quy tắc như các mẫu regex, khớp từ khóa hoặc kiểm tra tường minh. Tốc độ nhanh, có thể dự đoán được và tối ưu chi phí, nhưng có thể bỏ sót các vi phạm tinh vi.
* **Guardrail dựa trên model**: Sử dụng các LLM hoặc bộ phân loại để đánh giá nội dung bằng cách hiểu ngữ nghĩa. Nắm bắt được những vấn đề tinh vi mà các quy tắc có thể bỏ qua, nhưng tốc độ chậm hơn và tốn kém hơn.

LangChain cung cấp cả guardrail tích hợp sẵn (ví dụ: [phát hiện PII](https://docs.langchain.com/oss/python/langchain/guardrails#pii-detection), [human-in-the-loop](https://docs.langchain.com/oss/python/langchain/guardrails#human-in-the-loop)) và một hệ thống middleware linh hoạt để xây dựng các guardrail tùy chỉnh bằng cả hai phương pháp trên.

## Guardrail tích hợp sẵn

### Phát hiện PII

LangChain cung cấp middleware tích hợp sẵn để phát hiện và xử lý thông tin nhận dạng cá nhân (PII) trong các cuộc hội thoại. Middleware này có thể phát hiện các loại PII phổ biến như email, thẻ tín dụng, địa chỉ IP và nhiều loại khác.

Middleware phát hiện PII rất hữu ích cho các trường hợp như ứng dụng y tế và tài chính có yêu cầu tuân thủ quy định, các agent chăm sóc khách hàng cần làm sạch log, và nhìn chung là bất kỳ ứng dụng nào xử lý dữ liệu nhạy cảm của người dùng.

Middleware PII hỗ trợ nhiều chiến lược để xử lý các PII được phát hiện:

| Chiến lược | Mô tả | Ví dụ |
| -------- | --------------------------------------- | --------------------- |
| `redact` | Thay thế bằng `[REDACTED_{PII_TYPE}]` | `[REDACTED_EMAIL]` |
| `mask` | Che khuất một phần (ví dụ: 4 số cuối) | `****-****-****-1234` |
| `hash` | Thay thế bằng mã băm tất định | `a8f5f167...` |
| `block` | Ném ra exception khi phát hiện | Thông báo lỗi |

<div class="alert alert-info">

Khi đặt `apply_to_output=True`, `PIIMiddleware` cũng sẽ bôi đen (redact) đầu ra được truyền theo luồng - bao gồm text delta, đối số của tool-call, đầu ra của tool và các state snapshot - thông qua một stream transformer đã được đăng ký. Yêu cầu `langchain>=1.3.2`. Xem [Đăng ký transformer trên middleware](https://docs.langchain.com/oss/python/langchain/event-streaming#register-transformers-on-middleware).

</div>

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import PIIMiddleware


agent = create_agent(
    model="gpt-5.5",
    tools=[customer_service_tool, email_tool],
    middleware=[
        # Bôi đen (redact) email trong đầu vào của người dùng trước khi gửi tới model
        PIIMiddleware(
            "email",
            strategy="redact",
            apply_to_input=True,
        ),
        # Che khuất thẻ tín dụng trong đầu vào của người dùng
        PIIMiddleware(
            "credit_card",
            strategy="mask",
            apply_to_input=True,
        ),
        # Chặn API key - ném ra lỗi nếu phát hiện
        PIIMiddleware(
            "api_key",
            detector=r"sk-[a-zA-Z0-9]{32}",
            strategy="block",
            apply_to_input=True,
        ),
    ],
)

# Khi người dùng cung cấp PII, nó sẽ được xử lý theo chiến lược đã định
result = agent.invoke({
    "messages": [{"role": "user", "content": "Email của tôi là john.doe@example.com và thẻ của tôi là 5105-1051-0510-5100"}]
})

**Các loại PII tích hợp sẵn:**

* `email` - Địa chỉ email
* `credit_card` - Số thẻ tín dụng (Xác thực bằng thuật toán Luhn)
* `ip` - Địa chỉ IP
* `mac_address` - Địa chỉ MAC
* `url` - Đường dẫn URL

**Tùy chọn cấu hình:**

| Tham số | Mô tả | Mặc định |
| ----------------------- | ---------------------------------------------------------------------- | ---------------------- |
| `pii_type` | Loại PII cần phát hiện (tích hợp sẵn hoặc tùy chỉnh) | Bắt buộc |
| `strategy` | Cách xử lý PII được phát hiện (`"block"`, `"redact"`, `"mask"`, `"hash"`) | `"redact"` |
| `detector` | Hàm detector tùy chỉnh hoặc mẫu regex | `None` (sử dụng loại tích hợp sẵn) |
| `apply_to_input` | Kiểm tra tin nhắn của người dùng trước lệnh gọi model | `True` |
| `apply_to_output` | Kiểm tra tin nhắn của AI sau lệnh gọi model | `False` |
| `apply_to_tool_results` | Kiểm tra các tin nhắn chứa kết quả của tool sau khi thực thi | `False` |

Xem [tài liệu về middleware](https://docs.langchain.com/oss/python/langchain/middleware#pii-detection) để biết thông tin chi tiết về các tính năng phát hiện PII.

### Human-in-the-loop

LangChain cung cấp middleware tích hợp sẵn nhằm yêu cầu sự phê duyệt từ con người trước khi thực thi các thao tác nhạy cảm. Đây là một trong những guardrail hiệu quả nhất cho các quyết định mang tính rủi ro cao.

Middleware human-in-the-loop rất hữu ích cho các trường hợp như giao dịch và chuyển khoản tài chính, xóa hoặc sửa đổi dữ liệu trên môi trường production, gửi thông tin liên lạc cho các bên bên ngoài, và bất kỳ thao tác nào có tác động lớn đến nghiệp vụ.

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command


agent = create_agent(
    model="gpt-5.5",
    tools=[search_tool, send_email_tool, delete_database_tool],
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                # Yêu cầu phê duyệt đối với các thao tác nhạy cảm
                "send_email": True,
                "delete_database": True,
                # Tự động phê duyệt các thao tác an toàn
                "search": False,
            }
        ),
    ],
    # Lưu trữ state xuyên suốt các lần ngắt quãng (interrupt)
    checkpointer=InMemorySaver(),
)

# Human-in-the-loop yêu cầu một thread ID để lưu trữ dữ liệu
config = {"configurable": {"thread_id": "some_id"}}

# Agent sẽ tạm dừng và chờ phê duyệt trước khi thực thi các tool nhạy cảm
result = agent.invoke(
    {"messages": [{"role": "user", "content": "Gửi email cho nhóm"}]},
    config=config
)

result = agent.invoke(
    Command(resume={"decisions": [{"type": "approve"}]}),
    config=config  # Cùng một thread ID để tiếp tục cuộc trò chuyện đã bị tạm dừng
)

<div class="alert alert-success">

Xem [tài liệu về human-in-the-loop](https://docs.langchain.com/oss/python/langchain/human-in-the-loop) để biết thông tin chi tiết về cách triển khai các luồng phê duyệt.

</div>

## Guardrail tùy chỉnh

Đối với các guardrail phức tạp hơn, bạn có thể tạo middleware tùy chỉnh chạy trước hoặc sau khi agent thực thi. Điều này giúp bạn có toàn quyền kiểm soát đối với logic xác thực, lọc nội dung và kiểm tra an toàn.

### Guardrail chạy trước agent

Sử dụng các hook "before agent" để xác thực các yêu cầu ngay tại thời điểm bắt đầu mỗi lần gọi. Tính năng này hữu ích cho các bước kiểm tra ở cấp độ phiên (session-level) như xác thực quyền (authentication), giới hạn tốc độ (rate limiting) hoặc chặn các yêu cầu không phù hợp trước khi bắt đầu bất kỳ quá trình xử lý nào.

In [ ]:
from typing import Any

from langchain.agents.middleware import AgentMiddleware, AgentState, hook_config
from langgraph.runtime import Runtime

class ContentFilterMiddleware(AgentMiddleware):
    """Guardrail tất định: Chặn các yêu cầu chứa từ khóa bị cấm."""

    def __init__(self, banned_keywords: list[str]):
        super().__init__()
        self.banned_keywords = [kw.lower() for kw in banned_keywords]

    @hook_config(can_jump_to=["end"])
    def before_agent(self, state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
        # Lấy tin nhắn đầu tiên của người dùng
        if not state["messages"]:
            return None

        first_message = state["messages"][0]
        if first_message.type != "human":
            return None

        content = first_message.content.lower()

        # Kiểm tra các từ khóa bị cấm
        for keyword in self.banned_keywords:
            if keyword in content:
                # Chặn thực thi trước khi bắt đầu xử lý
                return {
                    "messages": [{
                        "role": "assistant",
                        "content": "Tôi không thể xử lý các yêu cầu chứa nội dung không phù hợp. Vui lòng diễn đạt lại yêu cầu của bạn."
                    }],
                    "jump_to": "end"
                }

        return None

# Sử dụng guardrail tùy chỉnh
from langchain.agents import create_agent

agent = create_agent(
    model="gpt-5.5",
    tools=[search_tool, calculator_tool],
    middleware=[
        ContentFilterMiddleware(
            banned_keywords=["hack", "exploit", "malware"]
        ),
    ],
)

# Yêu cầu này sẽ bị chặn trước khi quá trình xử lý diễn ra
result = agent.invoke({
    "messages": [{"role": "user", "content": "Làm cách nào để hack vào một cơ sở dữ liệu?"}]
})

### Guardrail chạy sau agent

Sử dụng các hook "after agent" để xác thực đầu ra cuối cùng trước khi trả về cho người dùng. Tính năng này hữu ích cho các bước kiểm tra an toàn dựa trên model, xác thực chất lượng hoặc rà soát tính tuân thủ cuối cùng trên toàn bộ phản hồi của agent.

In [ ]:
from langchain.agents.middleware import AgentMiddleware, AgentState, hook_config
from langgraph.runtime import Runtime
from langchain.messages import AIMessage
from langchain.chat_models import init_chat_model
from typing import Any

class SafetyGuardrailMiddleware(AgentMiddleware):
    """Guardrail dựa trên model: Sử dụng LLM để đánh giá mức độ an toàn của phản hồi."""

    def __init__(self):
        super().__init__()
        self.safety_model = init_chat_model("gpt-5.4-mini")

    @hook_config(can_jump_to=["end"])
    def after_agent(self, state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
        # Lấy phản hồi AI cuối cùng
        if not state["messages"]:
            return None

        last_message = state["messages"][-1]
        if not isinstance(last_message, AIMessage):
            return None

        # Sử dụng model để đánh giá an toàn
        safety_prompt = f"""Đánh giá xem phản hồi này có an toàn và phù hợp hay không.
        Chỉ trả lời 'SAFE' (AN TOÀN) hoặc 'UNSAFE' (KHÔNG AN TOÀN).

        Phản hồi: {last_message.content}"""

        result = self.safety_model.invoke([{"role": "user", "content": safety_prompt}])

        if "UNSAFE" in result.content:
            last_message.content = "Tôi không thể cung cấp phản hồi đó. Vui lòng diễn đạt lại yêu cầu của bạn."

        return None

# Sử dụng guardrail an toàn
from langchain.agents import create_agent

agent = create_agent(
    model="gpt-5.5",
    tools=[search_tool, calculator_tool],
    middleware=[SafetyGuardrailMiddleware()],
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Làm cách nào để chế tạo thuốc nổ?"}]
})

### Kết hợp nhiều guardrail

Bạn có thể kết hợp nhiều guardrail lại với nhau bằng cách thêm chúng vào mảng middleware. Chúng sẽ thực thi theo đúng thứ tự, cho phép bạn xây dựng một lớp bảo vệ đa tầng:

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import PIIMiddleware, HumanInTheLoopMiddleware

agent = create_agent(
    model="gpt-5.5",
    tools=[search_tool, send_email_tool],
    middleware=[
        # Lớp 1: Bộ lọc đầu vào tất định (chạy trước agent)
        ContentFilterMiddleware(banned_keywords=["hack", "exploit"]),

        # Lớp 2: Bảo vệ PII (chạy trước và sau model)
        PIIMiddleware("email", strategy="redact", apply_to_input=True),
        PIIMiddleware("email", strategy="redact", apply_to_output=True),

        # Lớp 3: Phê duyệt từ con người đối với các tool nhạy cảm
        HumanInTheLoopMiddleware(interrupt_on={"send_email": True}),

        # Lớp 4: Kiểm tra an toàn dựa trên model (chạy sau agent)
        SafetyGuardrailMiddleware(),
    ],
)

## Các tài nguyên bổ sung

* [Tài liệu về Middleware](https://docs.langchain.com/oss/python/langchain/middleware) - Hướng dẫn chi tiết về middleware tùy chỉnh
* [Tài liệu tham khảo API Middleware](https://reference.langchain.com/python/langchain/middleware/?_gl=1*ptlsdu*_gcl_au*MjA5MzEyMzM1NS4xNzg3MTg5OTgx*_ga*MTg4Njg5NDgwMS4xNzcwMzYxMzE1*_ga_47WX3HKKY2*czE3ODkzNzk1NzkkbzkyJGcwJHQxNzg5Mzc5NTc5JGo2MCRsMCRoMA..) - Hướng dẫn chi tiết về middleware tùy chỉnh
* [Human-in-the-loop](https://docs.langchain.com/oss/python/langchain/human-in-the-loop) - Bổ sung bước kiểm duyệt từ con người cho các thao tác nhạy cảm
* [Kiểm thử agent](https://docs.langchain.com/oss/python/langchain/test) - Các chiến lược để kiểm thử cơ chế an toàn